
# 26 — Modern LLM RL and Preference Optimization

## 01. Critic-Free Advantage Estimation

### Motivation

PPO typically estimates advantages using a learned value
function. Can we avoid training a Critic?

For a fixed prompt, sample G responses independently from
the same policy and score their complete outcomes.

GRPO uses group-normalized rewards:

$$
A_i^{GRPO} =
\frac{R_i-\bar R}{\operatorname{std}(R)+\epsilon}
$$

RLOO uses a leave-one-out baseline:

$$
A_i^{RLOO} =
R_i-\frac{1}{G-1}\sum_{j\ne i}R_j
$$

The two methods require no learned Critic, but their
gradient estimators are not identical.

This section studies only advantage estimation.
PPO-style clipping and token-level optimization come next.


In [1]:
import torch

# (P=2, G=3): three sampled responses for each of two prompts
rewards = torch.tensor([
    [1.0, 0.0, 0.0],  # One correct response
    [0.0, 0.0, 0.0],  # All responses incorrect
])

group_size = rewards.shape[1]

# (P=2, 1): mean and population std within each prompt
group_mean = rewards.mean(dim=1, keepdim=True)
group_std = rewards.std(dim=1, unbiased=False, keepdim=True)

# (P=2, G=3): GRPO group-normalized advantages
grpo_advantages = (rewards - group_mean) / group_std.clamp_min(1e-8)

# (P=2, G=3): average reward of OTHER responses
loo_baselines = (rewards.sum(dim=1, keepdim=True) - rewards) / (group_size - 1)

# (P=2, G=3): RLOO advantages
rloo_advantages = rewards - loo_baselines

print("GRPO advantages:\n", grpo_advantages)
print("RLOO advantages:\n", rloo_advantages)

# No reward contrast -> no group-relative learning signal.
assert torch.all(grpo_advantages[1] == 0)
assert torch.all(rloo_advantages[1] == 0)


GRPO advantages:
 tensor([[ 1.4142, -0.7071, -0.7071],
        [ 0.0000,  0.0000,  0.0000]])
RLOO advantages:
 tensor([[ 1.0000, -0.5000, -0.5000],
        [ 0.0000,  0.0000,  0.0000]])


In [2]:
p = torch.tensor(0.3, dtype=torch.float64)

# (N=4, G=2): all possible pairs of Bernoulli samples
actions = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float64)

# (N=4,): exact probabilities of the four sample pairs
pair_probs = torch.where(actions == 1, p, 1 - p).prod(dim=1)

# (N=4, G=2): score-function gradient d log pi(y) / d theta
score_gradients = actions - p

# (N=4, G=2): group-centered advantages
centered = actions - actions.mean(dim=1, keepdim=True)

# (N=4, G=2): leave-one-out advantages
loo = actions - actions.flip(dims=[1])

# (): exact expected policy-gradient estimators
centered_gradient = (pair_probs * (centered * score_gradients).mean(dim=1)).sum()

loo_gradient = (pair_probs * (loo * score_gradients).mean(dim=1)).sum()

# (): exact gradient of E[R] for this Bernoulli policy
true_gradient = p * (1 - p)

print("True gradient:", true_gradient.item())
print("Group-centered:", centered_gradient.item())
print("RLOO:", loo_gradient.item())

torch.testing.assert_close(loo_gradient, true_gradient)
torch.testing.assert_close(centered_gradient, 0.5 * true_gradient)


True gradient: 0.21
Group-centered: 0.105
RLOO: 0.21



### Critic-Free Advantage Estimation — Takeaways

A learned Critic is not the only way to construct a baseline.

Sampling multiple responses to the same prompt allows us
to compare rewards within a group.

GRPO uses group-centered and standardized outcome rewards.

RLOO uses the mean reward of the other samples, excluding
the current response from its own baseline.

Under independent sampling, the RLOO baseline preserves
the expected REINFORCE policy gradient.

Simple group-mean centering instead introduces a fixed
(G-1)/G scaling of the expected gradient.

Standard-deviation normalization changes this analysis.

When all responses receive identical rewards, their
group-relative advantages are zero.

Next: turn group-relative advantages into a token-level
PPO-style objective and derive GRPO.
